# rmul-scalar-tensor-mix — worked example 2: __add__ and __radd__ for a wrapper

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `rmul-scalar-tensor-mix`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

Addition is commutative, so a wrapper supporting `5 + w` needs `__radd__` to catch the case where the left operand (an int) returns `NotImplemented`. Like multiplication, `__radd__` may delegate to `__add__`. This is the same reflected-op mechanism as `__rmul__`.

## Worked solution

`Accum` wraps a scalar `value`. `__add__` adds either another `Accum`'s value or a raw number. `__radd__` delegates to `__add__` because `a + b == b + a` for numbers. This makes `sum([Accum(1), Accum(2)])` work: Python's `sum` starts with integer `0` and computes `0 + Accum(1)`, which dispatches to `Accum.__radd__`. We print the result of `sum` over a list of `Accum`s to show the reflected op enabling left-int addition.

In [ ]:
class Accum:
    def __init__(self, value):
        self.value = float(value)
    def __add__(self, other):
        ov = other.value if isinstance(other, Accum) else other
        return Accum(self.value + ov)
    def __radd__(self, other):
        return self.__add__(other)
    def __repr__(self):
        return f'Accum({self.value})'


items = [Accum(1), Accum(2), Accum(3)]
total = sum(items)  # starts from int 0 -> uses __radd__
print('sum:', total)
print('value:', total.value)